In [22]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import re

import kagglehub 
from kagglehub import KaggleDatasetAdapter

import spacy
from sklearn.feature_extraction import text

In [23]:
dataset_handle = "priyangshumukherjee/mental-health-text-classification-dataset"

# The main unbalanced training data
df_unbalanced = kagglehub.dataset_load(
  KaggleDatasetAdapter.PANDAS,
  dataset_handle,
  "mental_heath_unbanlanced.csv"
)

print("Unbalanced Data Loaded. Shape:", df_unbalanced.shape)
print(df_unbalanced.columns)
print(df_unbalanced.head())

Unbalanced Data Loaded. Shape: (49612, 3)
Index(['Unique_ID', 'text', 'status'], dtype='object')
   Unique_ID                                               text   status
0        0.0                                         oh my gosh  Anxiety
1        1.0  trouble sleeping, confused mind, restless hear...  Anxiety
2        2.0  All wrong, back off dear, forward doubt. Stay ...  Anxiety
3        3.0  I've shifted my focus to something else but I'...  Anxiety
4        4.0  I'm restless and restless, it's been a month n...  Anxiety


In [24]:
class_counts = df_unbalanced['status'].value_counts()
class_pct = df_unbalanced['status'].value_counts(normalize=True) * 100

print("Class Distribution:")
for label, count in class_counts.items():
    print(f"{label}: {count} ({class_pct[label]:.2f}%)")

print("\nMissing values:")
print(df_unbalanced.isnull().sum())

Class Distribution:
Normal: 18391 (37.07%)
Depression: 14506 (29.24%)
Suicidal: 11212 (22.60%)
Anxiety: 5503 (11.09%)

Missing values:
Unique_ID    9600
text            0
status          0
dtype: int64


In [25]:
duplicate_count = df_unbalanced.duplicated().sum()
print(f"Duplicate across the whole rows: {duplicate_count}")

text_duplicates = df_unbalanced.duplicated(subset=['text']).sum()
print(f"Duplicate text entries: {text_duplicates}")

if text_duplicates > 0:
    print("Some duplicate entries:")
    display(df_unbalanced[df_unbalanced.duplicated(subset=['text'], keep=False)].sort_values(by='text').head(5))

df_unbalanced = df_unbalanced.drop_duplicates(subset=['text'], keep='first')
print(f"\nDataset size after removing duplicates: {df_unbalanced.shape[0]}")

Duplicate across the whole rows: 0
Duplicate text entries: 667
Some duplicate entries:


,Unique_ID,text,status
39998,53027.0,"""Buy Friends"" comment? My mother doesn't have ...",Anxiety
39233,52041.0,"""Buy Friends"" comment? My mother doesn't have ...",Anxiety
18,18.0,"""No regrets or grudges/angry at things that ha...",Anxiety
307,309.0,"""No regrets or grudges/angry at things that ha...",Anxiety
500,504.0,"""No regrets or grudges/angry at things that ha...",Anxiety



Dataset size after removing duplicates: 48945


In [28]:
nlp = spacy.load("en_core_web_sm", disable=['parser', 'ner'])

KEEP_PRONOUNS = {'i','me','my','myself','we','our','ours'}

CONTRACTION_MAP = {
    "isn't": "is not", "aren't": "are not", "can't": "cannot", 
    "can't've": "cannot have", "could've": "could have", "couldn't": "could not",
    "didn't": "did not", "doesn't": "does not", "don't": "do not",
    "hadn't": "had not", "hasn't": "has not", "haven't": "have not",
    "he'd": "he would", "he'll": "he will", "he's": "he is",
    "i'd": "i would", "i'll": "i will", "i'm": "i am", "i've": "i have",
    "it's": "it is", "let's": "let us", "mightn't": "might not",
    "mustn't": "must not", "shan't": "shall not", "she'd": "she would",
    "she'll": "she will", "she's": "she is", "shouldn't": "should not",
    "they'd": "they would", "they'll": "they will", "they're": "they are",
    "they've": "they have", "weren't": "were not", "what'll": "what will",
    "what're": "what are", "what's": "what is", "what've": "what have",
    "where's": "where is", "who'd": "who would", "who'll": "who will",
    "who're": "who are", "who's": "who is", "who've": "who have",
    "won't": "will not", "wouldn't": "would not", "you'd": "you would",
    "you'll": "you will", "you're": "you are", "you've": "you have"
}

def expand_contractions(text, mapping):
    pattern = re.compile(r'\b(' + '|'.join(mapping.keys()) + r')\b')
    return pattern.sub(lambda x: mapping[x.group()], text)

def preprocess_text(text):
    text = text.lower()

    text = expand_contractions(text, CONTRACTION_MAP)

    text = re.sub(r'http\S+|www\S+|https\S+', '', text, flags=re.MULTILINE)
    text = re.sub(r'\@\w+|\#','', text)
    text = re.sub(r'[^a-z\s]', '', text)

    doc = nlp(text)

    tokens = []
    for token in doc:
        if token.text in KEEP_PRONOUNS:
            tokens.append(token.text)
        elif not token.is_stop and not token.is_space:
            tokens.append(token.lemma_)
    return " ".join(tokens)

print("Starting Preprocessing ... ")
df_unbalanced['cleaned_text'] = df_unbalanced['text'].apply(preprocess_text)
print("Preprocessing Complete")

df_unbalanced.to_csv('mental_health_preprocessed.csv', index=False)

Starting Preprocessing ... 
Preprocessing Complete
